# Quick Start

This page shows the first practical way to use CellPainting-Claw without pretending that every run starts from the same place.

Normal use has **two stages**:

- **stage or inspect inputs**
- **run skills on a local Cell Painting workspace**

The first stage can start from the Cell Painting Gallery. The second stage starts from a local workspace that contains the image and backend assets the skills need.

This notebook-style page shows both with **real recorded outputs**:

- a real Gallery inspection / planning / bounded download example
- a real downstream run on the repository demo workspace

The downstream run uses the repository demo config so the commands stay reproducible in public documentation. For your own project, you would replace that config with one that points at your own local workspace after input staging.


## Install

From the repository root:


In [ ]:
conda env create -f environment/cellpainting-claw.environment.yml
conda activate cellpainting-claw
pip install -e .[data-access,deepprofiler]


## Run Variables

The page below uses only **two working roots**:

- `DATA_ROOT` for input staging examples
- `RUN_ROOT` for the downstream skill run


In [ ]:
CONFIG=configs/project_config.demo.json
DATA_ROOT=demo/workspace/outputs/quick_start_data
RUN_ROOT=demo/workspace/outputs/quick_start_run


## Input Staging

If your data already exists in a local Cell Painting workspace, you can skip this section and move directly to segmentation or profiling.

If your inputs still live in the Cell Painting Gallery, the normal entry sequence is:

1. inspect configured sources
2. plan a download
3. download a bounded slice into local storage

The three commands below were run against the live Cell Painting Gallery with the demo config.


### Inspect Configured Sources


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill data-inspect-availability \
  --output-dir "$DATA_ROOT/01_inspect"


{
  "dataset_id": "cpg0016-jump",
  "gallery_dataset_count": 42,
  "gallery_source_count": 14,
  "first_sources": [
    "source_1",
    "source_10",
    "source_11",
    "source_13",
    "source_15"
  ],
  "required_packages_ok": true
}


This confirms that the configured Gallery access is usable, that `cpg0016-jump` is the default dataset in the demo config, and that `source_4` is one of the available sources.


### Plan A Download


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill data-plan-download \
  --dataset-id cpg0016-jump \
  --source-id source_4 \
  --max-files 4 \
  --output-dir "$DATA_ROOT/02_plan"


{
  "resolved_dataset_id": "cpg0016-jump",
  "resolved_source_id": "source_4",
  "resolved_prefix": "cpg0016-jump/source_4/",
  "step_count": 1,
  "max_files": 4
}


The planning step is useful when you want to confirm the resolved Gallery prefix before downloading a large source.


### Download A Small Local Input Slice


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill data-download \
  --dataset-id cpg0016-jump \
  --source-id source_4 \
  --subprefix workspace/load_data_csv/2021_04_26_Batch1/BR00117035 \
  --output-dir "$DATA_ROOT/03_download_small"


{
  "prefix": "cpg0016-jump/source_4/workspace/load_data_csv/2021_04_26_Batch1/BR00117035/",
  "matched_object_count": 2,
  "downloaded_count": 2,
  "object_keys": [
    ".../load_data.csv",
    ".../load_data_with_illum.csv"
  ]
}


This is intentionally a **small, bounded example**. It shows the real download path without turning the page into a long-running full-source transfer.

After this stage, the normal next step is to point your project config at a local workspace and start running the downstream skills.


## Local Workflow Run

The remaining steps use the repository demo workspace as the local input example. This is the part that corresponds to everyday skill execution once your inputs and backend assets are already local.

The recorded run below starts from local demo assets, writes segmentation outputs, exports single-cell crops, builds a DeepProfiler project, runs the model, collects tables, and summarizes the result.


### Segmentation Artifacts


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill cp-extract-segmentation-artifacts \
  --output-dir "$RUN_ROOT/01_segmentation"


{
  "derived_cppipe": "CPJUMP1_analysis_mask_export.cppipe",
  "load_data_rows": 2,
  "image_rows": 2,
  "cells_rows": 4,
  "nuclei_rows": 4
}


This step creates the main workflow root for the rest of the page. It also writes the **derived CellProfiler `.cppipe`** actually used for the run, which makes the chosen segmentation configuration inspectable.


### Single-Cell Crops


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill crop-export-single-cell-crops \
  --workflow-root "$RUN_ROOT/01_segmentation" \
  --output-dir "$RUN_ROOT/02_crops_masked"


{
  "crop_count": 4,
  "crop_ids": [
    "BR00000001_A01_s1_cell0001",
    "BR00000001_A01_s1_cell0002",
    "BR00000001_A02_s1_cell0001",
    "BR00000001_A02_s1_cell0002"
  ]
}


At this point the workflow has moved from field-level outputs to cell-level outputs.


### DeepProfiler Inputs


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill dp-export-deep-feature-inputs \
  --workflow-root "$RUN_ROOT/01_segmentation" \
  --output-dir "$RUN_ROOT/03_dp_inputs"


{
  "field_count": 2,
  "location_file_count": 2,
  "total_nuclei": 4,
  "channels": ["dna", "rna", "er", "agp", "mito"]
}


This is the bridge from segmentation outputs to model-ready DeepProfiler metadata.


### DeepProfiler Project


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill dp-build-deep-feature-project \
  --workflow-root "$RUN_ROOT/01_segmentation" \
  --export-root "$RUN_ROOT/03_dp_inputs" \
  --output-dir "$RUN_ROOT/04_dp_project"


{
  "experiment_name": "cell_painting_cnn_demo",
  "field_count": 2,
  "location_file_count": 2,
  "checkpoint_filename": "Cell_Painting_CNN_v1.hdf5",
  "image_format": "tiff"
}


The project build step reorganizes the export into the directory layout expected by DeepProfiler.


### DeepProfiler Inference


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill dp-run-deep-feature-model \
  --project-root "$RUN_ROOT/04_dp_project" \
  --output-dir "$RUN_ROOT/05_dp_run" \
  --gpu


2026-04-25 22:57:44.195590: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1616]
Created device /device:GPU:0 -> NVIDIA RTX PRO 6000 Blackwell Server Edition
2026-04-25 22:57:43.729639: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2027]
TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0.


The recorded run used a GPU-visible TensorFlow environment. For this tiny demo, that confirms environment wiring rather than demonstrating a speed advantage.


### DeepProfiler Tables


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill dp-collect-deep-features \
  --project-root "$RUN_ROOT/04_dp_project" \
  --output-dir "$RUN_ROOT/06_dp_features"


{
  "field_file_count": 2,
  "cell_count": 4,
  "well_count": 2,
  "feature_count": 672
}


This step turns raw DeepProfiler feature files into analysis-ready single-cell and well-level tables.


### DeepProfiler Summary


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill dp-summarize-deep-features \
  --single-cell-parquet-path "$RUN_ROOT/06_dp_features/deepprofiler_single_cell.parquet" \
  --well-aggregated-parquet-path "$RUN_ROOT/06_dp_features/deepprofiler_well_aggregated.parquet" \
  --output-dir "$RUN_ROOT/07_dp_summary"


{
  "cell_count": 4,
  "well_count": 2,
  "feature_count": 672,
  "top_feature_count": 50,
  "pca_components": 2
}


The summary step produces the first human-readable review layer on top of the collected embeddings, including a variability table and PCA outputs.


## Result Files

After this Quick Start, the most useful outputs to inspect are:

- `data_access_summary.json` for the configured source inventory
- `download_plan.json` for the resolved Gallery request
- `CPJUMP1_analysis_mask_export.cppipe` for the actual segmentation pipeline used
- `single_cell_manifest.csv` for exported crop records
- `field_metadata.csv` and per-field locations for the DeepProfiler bridge
- `deepprofiler_single_cell.parquet` and `deepprofiler_well_aggregated.parquet` for collected embeddings
- `profile_summary.json`, `top_variable_features.csv`, and `pca_plot.png` for the final review layer


## Next Pages

Continue with these pages after the first run:

- [CellPainting-Skills](../skills/index.md) for the full skill catalog
- [Command-Line Interface](../cli/index.md) for direct CLI usage
- [OpenClaw](../openclaw/index.md) for agent-mediated use of the same skills
